# Timing check (small notebook)
Run all cells: Runtime > Run all. It reloads the data, fits the seed-4 models, and times the detector twice (datasets in forward and in reverse order) to separate real differences from CPU noise. Allow Google Drive access when asked.

In [ ]:
import os, time, pickle, warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")

try:
    from google.colab import drive
except ImportError:
    drive = None                      # not running in Colab
if drive is not None:
    drive.mount("/content/drive")     # errors are shown, not hidden
    assert os.path.exists("/content/drive/MyDrive"), "Google Drive did not mount - re-run and click Allow"

# ---------------- CONFIG (edit here) ----------------
BASE = os.environ.get("DRELAB_BASE", "/content/drive/MyDrive/ColabNotebooks/DReLAB")
OUT = os.environ.get("REV_OUT", "/content/drive/MyDrive/ColabNotebooks/revision_results")
FAMILIES = {"CTU": ["Virut", "Neris", "Rbot", "Menti", "Murlo"],
            "BOTNET": ["Virut", "Neris", "Rbot", "Menti", "Murlo"],
            "CICIDS2018": ["Zeus_Ares"]}
ATTACKS = [("RF", "DDQN"), ("RF", "SARSA"), ("WnD", "DDQN"), ("WnD", "SARSA")]
SEEDS = list(range(int(os.environ.get("N_SEEDS", "5"))))
N_CHUNKS = int(os.environ.get("N_CHUNKS", "20"))       # position blocks
BUFFER = 0.02                                          # drop rows within 2% of a chunk edge
MAX_BENIGN = int(os.environ.get("MAX_BENIGN", "150000"))        # systematic (ordered) subsample of benign
MAX_TRAIN_PER_CLASS = int(os.environ.get("MAX_TRAIN", "40000"))
MAX_EVAL_PER_SET = int(os.environ.get("MAX_EVAL", "30000"))     # cap per val/test set (speed)
FAR_TARGET_PCT = 95                                    # threshold = 95th percentile of clean validation scores
BITS = 8                                               # quantization bits for the cleaned-up squeezer
DROP = {"Unnamed: 0", "Label", "label", "n_actions"}
os.makedirs(OUT, exist_ok=True)


In [ ]:
# ---------------- DATA ----------------
def with_pos(df):
    df = df.copy()
    df["_pos"] = (np.arange(len(df)) + 0.5) / len(df)
    return df

def load_dataset(ds):
    d = f"{BASE}/{ds}"
    benign = pd.read_csv(f"{d}/benign.csv")
    if len(benign) > MAX_BENIGN:                       # ordered subsample, keeps temporal structure
        benign = benign.iloc[np.linspace(0, len(benign) - 1, MAX_BENIGN).astype(int)]
    fams = FAMILIES[ds]
    orig = [with_pos(pd.read_csv(f"{d}/malicious/{f}.csv")) for f in fams]
    adv = {(o, a): [with_pos(pd.read_csv(f"{d}/{o}/{a}/{f}.csv")) for f in fams] for o, a in ATTACKS}
    benign = with_pos(benign)
    ref = [benign] + orig + [x for v in adv.values() for x in v]
    feats = [c for c in benign.columns
             if c not in DROP and c != "_pos"
             and all(c in r.columns for r in ref)
             and all(pd.api.types.is_numeric_dtype(r[c]) for r in ref)]
    def pool(frames):
        df = pd.concat(frames, ignore_index=True)
        X = df[feats].replace([np.inf, -np.inf], np.nan)
        X = X.fillna(X.median()).values.astype(np.float32)
        return X, df["_pos"].values
    data = {"benign": pool([benign]), "orig": pool(orig)}
    for k, v in adv.items():
        data[k] = pool(v)
    return data, feats

def split_roles(seed):
    rng = np.random.RandomState(seed)
    perm = rng.permutation(N_CHUNKS)
    n_tr, n_va = int(N_CHUNKS * 0.5), int(N_CHUNKS * 0.2)
    role = np.empty(N_CHUNKS, dtype=object)
    role[perm[:n_tr]] = "train"
    role[perm[n_tr:n_tr + n_va]] = "val"
    role[perm[n_tr + n_va:]] = "test"
    return role

def take(pool, role, split, rng, cap):
    X, pos = pool
    ch = np.minimum((pos * N_CHUNKS).astype(int), N_CHUNKS - 1)
    frac = pos * N_CHUNKS - ch
    m = (role[ch] == split) & (frac > BUFFER) & (frac < 1 - BUFFER)
    Xs = X[m]
    if len(Xs) > cap:
        Xs = Xs[rng.choice(len(Xs), cap, replace=False)]
    return Xs


In [ ]:
# ---- Fit the seed-4 models of each dataset (same procedure as the main run), then time them in two passes ----
TIMING_SEED = SEEDS[-1]
models = {}
for ds in FAMILIES:
    print("loading + fitting", ds)
    D, feats = load_dataset(ds)
    rng = np.random.RandomState(TIMING_SEED); role = split_roles(TIMING_SEED)
    Xb, Xo = take(D["benign"], role, "train", rng, 10**9), take(D["orig"], role, "train", rng, 10**9)
    n = min(len(Xb), len(Xo), MAX_TRAIN_PER_CLASS)
    Xb, Xo = Xb[rng.choice(len(Xb), n, replace=False)], Xo[rng.choice(len(Xo), n, replace=False)]
    Xtr = np.vstack([Xb, Xo]); ytr = np.r_[np.zeros(n), np.ones(n)]
    scaler = StandardScaler().fit(Xtr); Z = scaler.transform(Xtr).astype(np.float32)
    rf = RandomForestClassifier(n_estimators=150, max_depth=20, n_jobs=-1, random_state=TIMING_SEED).fit(Z, ytr)
    xgb = XGBClassifier(n_estimators=150, max_depth=8, learning_rate=0.1, eval_metric="logloss", random_state=TIMING_SEED).fit(Z, ytr)
    X = scaler.transform(take(D["benign"], role, "test", rng, 5000)).astype(np.float32)
    models[ds] = (rf, xgb, X, len(pickle.dumps(rf)) / 1e6, len(xgb.get_booster().save_raw()) / 1e6)
    del D, Xb, Xo, Xtr, Z

REPS = 15
def q(ts, n): a = np.array(ts) / n * 1e6; return np.median(a), np.percentile(a, 25), np.percentile(a, 75)
rows = []
for pass_name, order in [("forward", list(FAMILIES)), ("reverse", list(reversed(list(FAMILIES))))]:
    for ds in order:
        rf, xgb, X, rf_mb, xgb_mb = models[ds]
        fns = {"IDS Random Forest alone": lambda: rf.predict_proba(X),
               "Added XGBoost pass alone": lambda: xgb.predict_proba(X),
               "Ensemble disagreement detector": lambda: np.abs(rf.predict_proba(X)[:, 1] - xgb.predict_proba(X)[:, 1])}
        for f in fns.values(): f()
        ts = {k: [] for k in fns}
        for _ in range(REPS):
            for k, f in fns.items():
                t = time.perf_counter(); f(); ts[k].append(time.perf_counter() - t)
        old = rf.n_jobs; rf.n_jobs = 1; x1 = X[:1]
        fns1 = {"IDS Random Forest alone": lambda: rf.predict_proba(x1), "Added XGBoost pass alone": lambda: xgb.predict_proba(x1),
                "Ensemble disagreement detector": lambda: np.abs(rf.predict_proba(x1)[:, 1] - xgb.predict_proba(x1)[:, 1])}
        for f in fns1.values(): f()
        t1 = {k: [] for k in fns1}
        for _ in range(100):
            for k, f in fns1.items():
                t = time.perf_counter(); f(); t1[k].append(time.perf_counter() - t)
        rf.n_jobs = old
        for k in fns:
            med, lo, hi = q(ts[k], len(X))
            rows.append(dict(pass_=pass_name, dataset=ds, component=k, batch_ms_per_1000_median=round(med, 2),
                             q25=round(lo, 2), q75=round(hi, 2), single_flow_ms_median=round(np.median(t1[k]) * 1e3, 2)))
timing = pd.DataFrame(rows); timing.to_csv(f"{OUT}/timing_two_pass.csv", index=False)
print(timing.sort_values(["dataset", "component", "pass_"]).to_string(index=False))
print("\nModel sizes (MB), seed", TIMING_SEED, ":", {d: (round(m[3], 3), round(m[4], 3)) for d, m in models.items()})
